In [1]:
import os
import json
import numpy as np
from tqdm import tqdm
from langchain_community.embeddings import HuggingFaceEmbeddings

# ============================================================
# 1. CONFIGURATION & SETUP
# ============================================================
INPUT_FILE = "experiment_dataset.jsonl"
OUTPUT_FILE = "similarity_dataset.jsonl"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

print("Loading local Embedding model...")
local_embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

def compute_cosine_similarity(text1, text2):
    """Computes the cosine similarity between two text strings."""
    if not text1.strip() or not text2.strip():
        return 0.0
    
    # Generate vector embeddings
    v1 = np.array(local_embeddings.embed_query(text1))
    v2 = np.array(local_embeddings.embed_query(text2))
    
    # Calculate cosine similarity formula: (A . B) / (||A|| * ||B||)
    dot_product = np.dot(v1, v2)
    norm_v1 = np.linalg.norm(v1)
    norm_v2 = np.linalg.norm(v2)
    
    if norm_v1 == 0 or norm_v2 == 0:
        return 0.0
        
    return float(dot_product / (norm_v1 * norm_v2))

# ============================================================
# 2. CHECKPOINT ENGINE (Load Progress)
# ============================================================
processed_ids = set()

if os.path.exists(OUTPUT_FILE):
    print(f"🔄 Found existing progress file '{OUTPUT_FILE}'. Reading processed items...")
    with open(OUTPUT_FILE, 'r', encoding='utf-8') as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                existing_data = json.loads(line)
                if "id" in existing_data:
                    processed_ids.add(str(existing_data["id"]))
            except json.JSONDecodeError:
                print(f"⚠️ Warning: Line {line_num} in '{OUTPUT_FILE}' is corrupted. Skipping line.")
                
    print(f"⏮️ Found {len(processed_ids)} items already calculated. Skipping them automatically.\n")

# ============================================================
# 3. STREAM & CALCULATE SIMILARITY
# ============================================================
if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"❌ Input file '{INPUT_FILE}' not found!")

# Pre-calculate total rows for the progress bar
total_lines = 0
with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip(): 
            total_lines += 1

print(f"🚀 Running Retrieval Similarity Vector Engine...")

# Open output in append mode ('a') for safety
with open(INPUT_FILE, 'r', encoding='utf-8') as infile, \
     open(OUTPUT_FILE, 'a', encoding='utf-8') as outfile:
    
    progress_bar = tqdm(infile, total=total_lines, desc="📐 Vector Similarity", unit="item")
    
    for line_num, line in enumerate(progress_bar, 1):
        line = line.strip()
        if not line:
            continue
            
        try:
            data = json.loads(line)
            item_id = str(data.get("id"))
            
            # Checkpoint skip
            if item_id in processed_ids:
                continue
                
            rag_answer = data.get("rag_answer", "")
            
            # Skip if there's no answer generated yet
            if not rag_answer:
                continue
                
            progress_bar.set_postfix_str(f"ID: {item_id[:6]}")
            
            # Text Normalization: Flatten context/evidence into a single string
            raw_evidence = data.get("evidence", "")
            if isinstance(raw_evidence, list):
                context_str = " ".join([str(item) for item in raw_evidence])
            else:
                context_str = str(raw_evidence)
            
            # Compute math vector similarity
            similarity_score = compute_cosine_similarity(context_str, rag_answer)
            
            # Update dictionary and write immediately to disk
            data["retrail_similarity"] = similarity_score
            
            outfile.write(json.dumps(data, ensure_ascii=False) + "\n")
            outfile.flush()  # Force hard drive to save the line

        except json.JSONDecodeError:
            print(f"\n⚠️ Malformed JSON at line {line_num} of input file. Skipping...")
        except Exception as e:
            print(f"\n❌ Error processing item {data.get('id', 'Unknown')}: {e}. Skipping...")

print(f"\n🎉 Process complete! All semantic similarity scores saved in '{OUTPUT_FILE}'.")

Loading local Embedding model...


/var/folders/04/ycg4c8hs6tncxvm8l19zwkj80000gn/T/ipykernel_18931/4120305312.py:15: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  local_embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
/Users/sterinsaji/miniconda3/envs/rag_project/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5004.15it/s]


🚀 Running Retrieval Similarity Vector Engine...


📐 Vector Similarity: 100%|██████████| 519/519 [00:37<00:00, 13.81item/s, ID: c534ea]


🎉 Process complete! All semantic similarity scores saved in 'similarity_dataset.jsonl'.
